# GameTheory-23 : Trouver des failles légales — recherche bornée contre preuve Lean (Lexifina, 2026)

**Navigation** : [<< 22-Jeux stochastiques](GameTheory-22-Jeux-Stochastiques-Shapley-Python.ipynb) | [Index](README.md)

## Le droit comme espace de jeu

En novembre 2026, Adnan Yahya propose sur Lexifina (rubrique *Opinion*) de traiter les règles
fiscales américaines des retraites comme un **jeu formel** : des états (les comptes du
contribuable), des coups (les mouvements d'argent autorisés par le code), des gardes (les
conditions de légalité) — puis d'y chercher des **failles légales** (« loopholes ») par recherche
exhaustive bornée, et de **prouver en Lean** des théorèmes sur ce que ces failles permettent et
ne permettent pas. Dans leur modèle complet (5 comptes, 7 coups, limites 2025), une recherche
BFS de 6 coups de profondeur — 345 075 positions — retrouve mécaniquement le *backdoor Roth*
et le *mega backdoor* ($77 000 versés en Roth sans taxe) **sans qu'on les lui décrive**, et un
théorème de plafond prouve le plan trouvé optimal.

Ce carnet distille la méthode sur un **modèle jouet réduit** (3 comptes, 4 coups) : un BFS borné
en Python retrouve le plan optimal sans qu'on le lui décrive, puis le **compagnon Lean**
`LegalLoophole.Toy` (ajouté au lake `game_theory_lean` par ce même grain) prouve les deux
théorèmes qui donnent à ce plan son statut — il existe, et rien ne fait mieux.

**Références.**
- Adnan Yahya, « *Lean proof to search for US tax loopholes* », Lexifina, 09/11/2026, rubrique
  *Opinion* — un point de vue méthodologique, **pas une publication évaluée**. Post précédent du
  même auteur : « *auto-formalization of mathematics versus law* », Lexifina, 07/09/2026.
  Copie archivée au gisement bibliographique : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Légal\2026-11-09 - Yahya Adnan - Lean proof to search for US tax loopholes - Lexifina.pdf`.
- Code public : dépôt [`lexifina/global-law`](https://github.com/lexifina/global-law), dossier
  `lean-proofs/USTax/Retirement/` (`Rules.lean` + `Theorems.lean`, licence **CC0 1.0**, lu au
  commit `24036e9a`, toolchain v4.35.0, zéro dépendance). Notre jouet en est une **copie
  pédagogique déclarée** (échelle réduite, notre toolchain v4.33.0) — l'étude complète du dépôt
  et la voie d'intégration motivée (portage plutôt que dépendance Lake) sont consignées sur
  l'issue #25686.

**Prérequis.** La double lecture témoin/théorème des carnets
[Metajeux-04](Metajeux/Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb) et
[Metajeux-05](Metajeux/Metajeux-05-Temoins-Impossibilite-Python.ipynb), et du carnet
[Lean-25](../SymbolicAI/Lean/Lean-25-Coherence-et-Temoin.ipynb) ; le point de
vue « règles = programme à gardes » de la librairie `ProgramGames` du même lake. Aucun
prérequis fiscal : toutes les règles du jouet sont déclarées ci-dessous.

> **Avertissement.** Modèle pédagogique à montants symboliques (1 unité = 1 000 $) — ceci
> n'est pas un conseil fiscal, et le jouet écarte délibérément la plupart des règles réelles
> (voir §5).

## 1. Des règles fiscales à un jeu formel

Une règle fiscale dit « qui a le droit de déplacer combien, vers où, en payant quoi ». Chacun de
ces mots devient un champ d'état ou une garde de coup :

| Élément de la règle | Champ du jeu |
|---|---|
| Les comptes du contribuable | état `Account` : `cash`, `pretax`, `basis`, `roth` |
| Le plafond annuel de contribution | loi `max_contrib`, suivie par le champ `used` |
| La limite de revenu sur le Roth direct | garde `not income_restricted` du coup `direct` |
| La taxation au pro-rata d'une conversion | `convert_tax(n) = n·pretax / (pretax + basis)` au plancher |
| La légalité d'un mouvement | le coup renvoie un état, ou `None` s'il est illégal |

Le compte **traditionnel** est ventilé en deux parts : **pre-tax** (déduite à l'entrée, imposable
à la sortie) et **basis** (déjà taxée). C'est cette ventilation qui rend le pro-rata calculable —
et c'est elle que la faille du §2 exploite. Le point de vue « un coup est un programme à
gardes » est exactement celui de la librairie `ProgramGames`
(`game_theory_lean/ProgramGames/`) : là où ses agents se sondent mutuellement par des
programmes, ici les gardes portent sur des montants.

Les quatre coups du jouet, chacun dérivé d'une provision citée par l'article :

| Coup | Effet | Garde principale |
|---|---|---|
| `direct n` | cash → Roth, sans taxe | interdit si `income_restricted` |
| `deductible n` | cash → part pre-tax | budget annuel partagé |
| `nondeductible n` | cash → basis | budget annuel partagé |
| `convert n` | traditionnel → Roth | taxe pro-rata `convert_tax(n)` |

In [1]:
from dataclasses import dataclass

# --- Miroir Python exact du module Lean LegalLoophole.Toy -------------------
# 1 unite = 1 000 $ (montants symboliques). Le traditionnel est ventile en
# part pre-tax (deductible, taxee a la sortie) et basis (deja taxee).

@dataclass(frozen=True)
class TaxLaws:
    """Lois du jouet : budget annuel partage + restriction de revenu."""
    max_contrib: int          # plafond annuel des contributions
    income_restricted: bool   # True : contribution Roth directe interdite

@dataclass(frozen=True)
class Account:
    """Etat du contribuable : 3 comptes + compteur de budget annuel."""
    cash: int = 0    # argent neuf, deja taxe
    pretax: int = 0  # part pre-tax du traditionnel
    basis: int = 0   # part after-tax (deja taxee) du traditionnel
    roth: int = 0    # compte Roth (sorties non imposables)
    used: int = 0    # budget de contribution deja consomme

def convert_tax(s: Account, n: int) -> int:
    """Taxe d'une conversion de n : part pre-tax convertie, arrondie au plancher.

    Denominateur nul : impossible en pratique, car le garde n <= pretax + basis
    de `step` laisse n = 0, et en Lean 0 * pretax / 0 vaut 0 (Nat.div) -- le
    miroir Python rend le meme 0 explicitement.
    """
    denom = s.pretax + s.basis
    return n * s.pretax // denom if denom else 0

def step(laws, s, move):
    """Un coup legal : (etat suivant, taxe prelevee) ou None si illegal.

    Coups : ('direct', n) | ('deductible', n) | ('nondeductible', n) | ('convert', n).
    Gardes identiques a celles du module Lean.
    """
    kind, n = move
    if kind == 'direct':
        if (not laws.income_restricted and n <= laws.max_contrib
                and n + s.used <= laws.max_contrib and n <= s.cash):
            return (Account(s.cash - n, s.pretax, s.basis, s.roth + n, s.used + n), 0)
    elif kind == 'deductible':
        if (n <= laws.max_contrib and n + s.used <= laws.max_contrib
                and n <= s.cash):
            return (Account(s.cash - n, s.pretax + n, s.basis, s.roth, s.used + n), 0)
    elif kind == 'nondeductible':
        if (n <= laws.max_contrib and n + s.used <= laws.max_contrib
                and n <= s.cash):
            return (Account(s.cash - n, s.pretax, s.basis + n, s.roth, s.used + n), 0)
    elif kind == 'convert':
        tax = convert_tax(s, n)
        if (n <= s.pretax + s.basis and tax <= n and tax <= s.cash
                and tax <= s.pretax and n - tax <= s.basis):
            return (Account(s.cash - tax, s.pretax - tax, s.basis - (n - tax),
                            s.roth + n, s.used), tax)
    return None

def run(laws, s0, moves):
    """Execution d'une sequence de coups : (etat final, taxe totale) ou None."""
    s, total = s0, 0
    for mv in moves:
        res = step(laws, s, mv)
        if res is None:
            return None
        s, t = res
        total += t
    return (s, total)

# Verification rapide du miroir : le coup direct est interdit aux hauts
# revenus ; la sequence non-deductible puis conversion s'execute sans taxe
# sur un IRA vide.
LOIS = TaxLaws(max_contrib=7, income_restricted=True)
S0 = Account(cash=10, pretax=0, basis=0, roth=0, used=0)
print("coup direct, haut revenu :", step(LOIS, S0, ('direct', 7)))
print("backdoor sur IRA vide   :", run(LOIS, S0, [('nondeductible', 7), ('convert', 7)]))

coup direct, haut revenu : None
backdoor sur IRA vide   : (Account(cash=3, pretax=0, basis=0, roth=7, used=7), 0)


### Lecture du résultat

La sémantique est **exécutable** : `step` et `run` sont du code normal. Le coup direct est
refusé (`None`) sur une instance à haut revenu, et la séquence *non-déductible puis conversion*
s'exécute sans taxe sur un IRA vide. Ce miroir Python est **l'exact reflet du module Lean**
`LegalLoophole.Toy` (mêmes gardes, même arrondi au plancher) : tout ce que la recherche trouve
ci-dessous a vocation à être généralisé par un théorème du §3, pas seulement observé.

Un détail de modèle mérite l'œil : les gardes de `convert` exigent `tax <= n`, `tax <= pretax`
et `n - tax <= basis` — des conditions de solvabilité qui interdisent toute troncature silencieuse
des soustractions. Ce sont des **hypothèses déclarées** (démontrables depuis
`n <= pretax + basis` par l'arithmétique de la division au plancher) : le jouet ne simule pas
d'argent négatif.

## 2. La recherche bornée — trouver le plan sans le décrire

La question posée au moteur : *sur l'instance du jouet (plafond 7, haut revenu, 10 de cash,
IRA vide), quelle est la meilleure séquence légale d'au plus 4 coups ?* — où « meilleure »
veut dire : Roth final maximal, puis taxe totale minimale. Personne ne lui dit que la
contribution non déducible existe pour être convertie : le moteur énumère **tous** les coups
(les quatre sortes, tous les montants sous plafond), ne garde que les coups légaux (les gardes
de `step` tranchent), et déplie l'espace par profondeur — le BFS de l'article, réduit à
l'échelle du jouet.

Deux bornes déclarées : la **profondeur** (≤ 4 coups) et les **montants** (≤ `max_contrib`).
L'espace exploré est fini et petit ; tout ce que dit la recherche est vrai *à l'intérieur de ces
bornes*. Ce que la recherche ne peut pas dire — « existe-t-il un plan meilleur, plus profond,
avec d'autres montants ? » — fait l'objet des théorèmes du §3.

In [2]:
from collections import deque

def coups_possibles(laws, s):
    """Enumeration bornee des coups depuis s : chaque sorte de coup, tous les
    montants n = 1..plafond. Les gardes de `step` tranchent la legalite."""
    coups = []
    cap_contrib = min(laws.max_contrib - s.used, s.cash)
    for kind in ('direct', 'deductible', 'nondeductible'):
        for n in range(1, cap_contrib + 1):
            coups.append((kind, n))
    for n in range(1, s.pretax + s.basis + 1):
        coups.append(('convert', n))
    return coups

def meilleure_sequence(laws, s0, profondeur_max=4):
    """Recherche bornee (BFS par profondeur) : meilleure sequence legale de
    profondeur <= profondeur_max, au sens (roth final maximal, puis taxe
    totale minimale). Retourne (sequence, etat_final, taxe_totale).

    Exhaustive sur l'espace borne : la cle de deduplication est le couple
    (etat, taxe cumulee), donc aucun plan de l'espace n'est ecarte.
    """
    cle = lambda s, t: (s.roth, -t)
    meilleur = (cle(s0, 0), [], s0, 0)
    vus = {(s0, 0)}
    frontiere = deque([(s0, [], 0)])
    while frontiere:
        s, seq, taxe = frontiere.popleft()
        if cle(s, taxe) > meilleur[0]:
            meilleur = (cle(s, taxe), seq, s, taxe)
        if len(seq) >= profondeur_max:
            continue
        for mv in coups_possibles(laws, s):
            res = step(laws, s, mv)
            if res is None:
                continue
            s1, t1 = res
            if (s1, taxe + t1) in vus:
                continue
            vus.add((s1, taxe + t1))
            frontiere.append((s1, seq + [mv], taxe + t1))
    return meilleur[1], meilleur[2], meilleur[3]

In [3]:
seq, etat, taxe = meilleure_sequence(LOIS, S0, profondeur_max=4)
print("Plan trouve :")
for mv in seq:
    print("   ", mv)
print("Etat final :", etat)
print("Taxe totale :", taxe)

Plan trouve :
    ('nondeductible', 7)
    ('convert', 7)
Etat final : Account(cash=3, pretax=0, basis=0, roth=7, used=7)
Taxe totale : 0


### Lecture du résultat

La recherche a retrouvé, seule, le plan dit **« backdoor Roth IRA »** : contribuer en
**non déductible** (l'argent neuf entre en `basis`, déjà taxée — la limite de revenu ne
s'applique pas aux contributions traditionnelles), puis **convertir** (la `basis` se convertit
sans taxe supplémentaire : le pro-rata ne voit que du pre-tax, et il n'y en a pas). Résultat :
`roth = 7` — tout le budget annuel — pour `taxe = 0`. Le coup `direct`, interdit par la limite
de revenu, n'apparaît nulle part : la contrainte a été **contournée, pas violée** — chaque coup
du plan est légal pris isolément. C'est exactement le théorème 3 de l'article
(`income_limit_void` : la limite §408A est contournée par construction dans le modèle), à
l'échelle du jouet.

### Exercice 1 — le plafond est-il vraiment un plafond ?

Le plan trouvé met exactement `max_contrib = 7` en Roth. Est-ce le maximum **atteignable par
toute séquence bornée**, ou la recherche s'est-elle arrêtée trop tôt ? À vous de le vérifier
exhaustivement.

In [4]:
def plafond_atteignable(laws, s0, profondeur_max=4):
    """Renvoie (roth_maximal, taxe_du_meilleur_plan) atteints par TOUTE
    sequence legale de profondeur <= profondeur_max sur l'instance donnee.

    # Etape 1 : enumerer toute la frontiere (etat, taxe cumulee) comme dans
    #           meilleure_sequence -- sans s'arreter au premier optimum
    # Etape 2 : renvoyer le roth maximal rencontre, et la taxe minimale
    #           parmi les plans qui l'atteignent
    # Indice : chaque contribution (des trois sortes) consomme du budget
    #          `used` plafonne a max_contrib, et sur un IRA vide rien d'autre
    #          n'alimente le Roth -- attendu ici : (7, 0).
    """
    # TODO etudiant
    return None  # TODO etudiant

In [5]:
# Contre-experience : la MEME recherche sur un IRA qui contient deja 3 de pre-tax.
S0_SALE = Account(cash=10, pretax=3, basis=0, roth=0, used=0)
seq, etat, taxe = meilleure_sequence(LOIS, S0_SALE, profondeur_max=4)
print("IRA sale (pretax=3) -- plan trouve :")
for mv in seq:
    print("   ", mv)
print("Etat final :", etat)
print("Taxe totale :", taxe)

IRA sale (pretax=3) -- plan trouve :
    ('nondeductible', 7)
    ('convert', 10)
Etat final : Account(cash=0, pretax=0, basis=0, roth=10, used=7)
Taxe totale : 3


### Lecture du résultat

Sur un IRA qui contient déjà 3 de pre-tax, la recherche trouve un plan **en deux coups** : contribuer 7 en non déductible (le traditionnel contient alors 3 de pre-tax et 7 de basis), puis **tout convertir d'un seul coup**. La taxe pro-rata de cette conversion vaut 10 × 3 / (3 + 7) = 3 — exactement la part pre-tax de la conversion. Roth final : 10 = 3 + 7, qui dépasse le plafond annuel parce que les 3 pre-tax appartenaient déjà au contribuable ; taxe totale : 3, pas 0.

C'est la règle du **pro-rata** qui punit le backdoor naïf sur un IRA sale. Convertir seulement la part basis (7) coûterait 7 × 3 / (3 + 7) = 2 de taxe en laissant le Roth à 7 ; tout plan qui maximise le Roth doit bien finir par convertir les 3 de pre-tax — et les deux découpages possibles paient 3 au total. Le contre-témoin `#eval` du §3 calcule la variante à 2. L'article rapporte le même phénomène à l'échelle réelle : convertir $7 000 de pre-tax sans roll-in paie **$6 543** d'impôt dans leur modèle complet. En droit réel, l'agrégation se fait en fin d'année (formulaire 8606) : le jouet la calcule conversion par conversion — hypothèse déclarée, voir §5.

### Exercice 2 — le prix du pro-rata.

Le plan naïf coûte 2 sur notre instance sale ; que coûte-t-il quand l'IRA l'est encore plus ?

In [6]:
def taxe_backdoor_naif(pretax_initial, max_contrib=7, cash=10):
    """Taxe totale de la sequence FIXE [non-deductible max_contrib, convert
    max_contrib] sur un etat (cash, pretax_initial, basis=0, roth=0, used=0).
    Retourne la taxe totale (int), ou None si un coup est illegal.

    # Etape 1 : construire l'etat initial, executer la sequence via run()
    # Etape 2 : comparer pretax_initial = 1, 3, 5, 7 et verifier la formule
    #           max_contrib * pretax_initial / (pretax_initial + max_contrib)
    #           arrondie au plancher
    # Indice : la taxe se calcule sur l'etat APRES la contribution -- le
    #          denominateur du pro-rata est pretax_initial + max_contrib.
    """
    # TODO etudiant
    return None  # TODO etudiant

## 3. Le compagnon Lean — la même sémantique, prouvée

Le module `game_theory_lean/LegalLoophole/Toy.lean` (10ᵉ `lean_lib` du package
`game_theory_lean`, ajoutée par ce grain) écrit le même jouet en Lean 4 : **zéro dépendance**
(pas de Mathlib), compilable en quelques secondes. Mêmes gardes, même `convert_tax` au
plancher, même sémantique `step`/`run` sur `Option` — et des théorèmes que la recherche du §2
ne peut qu'effleurer. La preuve de compilation du module dans son contexte de lake est la
jambe CI `gametheory` (règle du dépôt : pas de build de lake local, la CI est la preuve) ; la
vérification bornée ci-dessous — un seul fichier, aucune dépendance — sert au carnet : elle
montre le témoin **calculer**.

In [7]:
from pathlib import Path

def racine_repo():
    """Localise la racine du depot en remontant depuis le repertoire de
    travail (robuste au cwd d'execution), sans afficher de chemin machine."""
    marqueurs = ['MyIA.AI.Notebooks', 'scripts', 'CLAUDE.md']
    for base in [Path.cwd(), *Path.cwd().parents]:
        if all((base / m).exists() for m in marqueurs):
            return base
    raise RuntimeError('racine du depot introuvable depuis le repertoire de travail')

TOY = (racine_repo() / 'MyIA.AI.Notebooks' / 'GameTheory' / 'game_theory_lean'
       / 'LegalLoophole' / 'Toy.lean')
lignes = TOY.read_text(encoding='utf-8').splitlines()
print(f"Module Toy.lean : {len(lignes)} lignes, 0 sorry, 0 axiome ad hoc, 0 import")
print()
print("Signatures des theoremes du module :")
for i, l in enumerate(lignes):
    if l.startswith('theorem '):
        print(f"{i+1:4d} | {l.rstrip()}")
print()
print("Le coup convert (garde de solvabilite complete) :")
for i, l in enumerate(lignes):
    if l.lstrip().startswith('| .convert n'):
        for j in range(i, min(i + 9, len(lignes))):
            print(f"{j+1:4d} | {lignes[j].rstrip()}")
        break

Module Toy.lean : 394 lignes, 0 sorry, 0 axiome ad hoc, 0 import

Signatures des theoremes du module :
 163 | theorem step_conservation (laws : TaxLaws) (s : Account) (m : Move)
 218 | theorem step_cash_floor (laws : TaxLaws) (s : Account) (m : Move)
 268 | theorem wealth_conservation (laws : TaxLaws) (s : Account) :
 298 | theorem cash_floor (laws : TaxLaws) (s : Account) (hu : s.used ≤ laws.maxContrib) :
 331 | theorem roth_ceiling (laws : TaxLaws) (s : Account) (hu : s.used ≤ laws.maxContrib) :
 348 | theorem backdoor_exists (laws : TaxLaws) (s : Account)
 364 | theorem backdoor_optimal (laws : TaxLaws) (s : Account)

Le coup convert (garde de solvabilite complete) :
 136 |   | .convert n =>
 137 |       let tax := convertTax s n
 138 |       if n ≤ s.pretax + s.basis ∧ tax ≤ n ∧ tax ≤ s.cash ∧ tax ≤ s.pretax ∧ n - tax ≤ s.basis then
 139 |         some ({ s with
 140 |                   cash := s.cash - tax
 141 |                   pretax := s.pretax - tax
 142 |                   

In [8]:
import os
import subprocess

repo = racine_repo()
# Chemin WSL du module : conversion D:\Dev\... -> /mnt/d/Dev/...
wsl_dir = ('/mnt/' + repo.drive.lower().rstrip(':')
           + (repo / 'MyIA.AI.Notebooks' / 'GameTheory' / 'game_theory_lean'
              / 'LegalLoophole').as_posix()[2:])
# Aucun nom de compte n'est code en dur : sans COURSIA_WSL_USER, on laisse WSL
# choisir l'utilisateur par defaut de la distro (celui qui porte elan).
wsl_distro = os.getenv("COURSIA_WSL_DISTRO", "Ubuntu")
wsl_user = os.getenv("COURSIA_WSL_USER")  # vide -> utilisateur par defaut
cmd = ["wsl", "-d", wsl_distro] + (["-u", wsl_user] if wsl_user else [])
cmd += ["--", "bash", "-lc",
        f'export PATH="$HOME/.elan/bin:$PATH"; cd "{wsl_dir}" && lean Toy.lean']
# Verification bornee (secondes, zero dependance) -- la compilation du module
# dans son contexte de lake reste la jambe CI `gametheory`.
res = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8',
                     errors='replace', timeout=120)
print("rc =", res.returncode)
print(res.stdout.rstrip())
temoins = ['cash := 3', 'cash := 1']
print("temoins #eval presents :", all(t in res.stdout for t in temoins))

rc = 0
some ({ cash := 3, pretax := 0, basis := 0, roth := 7, used := 7 }, 0)
some ({ cash := 1, pretax := 1, basis := 2, roth := 7, used := 7 }, 2)
temoins #eval presents : True


### Lecture du résultat

`rc = 0` : le module typechecke sans erreur ni `sorry`, et les deux `#eval` témoins calculent —
le backdoor sur IRA vide (taxe 0, Roth +7, budget épuisé) et le contre-témoin sur IRA sale
(pro-rata de 2, exactement `7 × 3 / (3 + 7)` au plancher). Les signatures listées plus haut sont
donc de **vraies propositions prouvées**, pas des commentaires :

- `wealth_conservation` — aucun coup ne crée d'argent : richesse totale + taxes versées au
  Trésor = invariant (jouet du théorème 1 de l'article, `pretax_conservation`) ;
- `cash_floor` puis `roth_ceiling` — pour **toute** séquence légale, de **toute profondeur**,
  depuis **tout état de départ** sous budget : le Roth final est au plus
  `roth + pretax + basis + max_contrib` de l'état initial. En particulier sur un IRA vide,
  aucun plan légal ne met plus de `max_contrib` en Roth, taxé ou pas (jouet du théorème 4,
  `roth_ceiling` de l'article) ;
- `backdoor_exists` et `backdoor_optimal` — le témoin [non-déductible, convertir] atteint
  **exactement** ce plafond sans payer un centime : le plan trouvé par la recherche est
  **optimal**, et c'est un théorème (jouet du théorème 3, `income_limit_void`).

### Exercice 3 — un cinquième coup : le retrait Roth.

Après 5 ans (ramenés à l'instant dans le jouet), le Roth se retire sans impôt. Que fait ce coup
au théorème de conservation ?

In [9]:
def step_avec_retrait(laws, s, move):
    """Miroir de step(), plus la branche ('withdraw', n) : retrait non
    imposable du Roth vers le cash. Retourne (Account, taxe) ou None.

    # Etape 1 : ajouter la branche : garde n <= s.roth, effet
    #           Account(s.cash + n, s.pretax, s.basis, s.roth - n, s.used),
    #           taxe 0 -- les quatre autres coups inchanges
    # Etape 2 : verifier sur un exemple que la richesse totale
    #           (cash + pretax + basis + roth + taxes payees) ne bouge pas
    # Indice : le retrait deplace de l'argent ENTRE deux champs du meme etat,
    #          comme `direct` -- en Lean, il suffirait d'ajouter la branche au
    #          `match` de step et un cas au `cases m` des deux lemmes par coup.
    """
    # TODO etudiant
    return None  # TODO etudiant

## 4. Témoin et théorème — la double lecture d'une faille

| Question | Recherche bornée (§2) | Théorème Lean (§3) |
|---|---|---|
| « Un plan sans taxe existe-t-il ? » | **oui** : un témoin concret, avec sa séquence | `backdoor_exists` : pour tout IRA vide et tout cash suffisant |
| « Peut-on faire mieux que 7 ? » | vérifié **sur l'espace borné** seulement | `roth_ceiling` : à **toute** profondeur, non — `backdoor_optimal` fait coïncider témoin et borne |
| « Et si l'IRA est sale ? » | la recherche compare les plans (nettoyer d'abord) | hors théorème du jouet : c'est une **expérience**, pas un théorème |

C'est la dualité travaillée aux carnets
[Metajeux-04](Metajeux/Metajeux-04-Chemin-Minimal-Robinson-Goforth-Python.ipynb) et
[Metajeux-05](Metajeux/Metajeux-05-Temoins-Impossibilite-Python.ipynb),
et au carnet [Lean-25](../SymbolicAI/Lean/Lean-25-Coherence-et-Temoin.ipynb) : un **témoin**
démontre la possibilité par l'exemple, un **théorème** démontre l'universel — et
`backdoor_optimal` est la **jonction** des deux : le témoin atteint exactement la borne, donc
il est optimal. La recherche seule ne pourrait jamais conclure à l'optimalité (il reste
toujours des séquences plus profondes que la borne) ; le théorème seul ne donne pas le plan
(il est existentiel). **La méthode de l'article est la paire** — et l'article en tire un
chiffre : son plan BFS atteint le plafond de $77 000, donc il est prouvé optimal dans le
modèle.

Le contraste avec la conception de mécanismes est net : un mécanisme **DSIC** (carnet
[GameTheory-16c](GameTheory-16c-Extraction-de-Revenu-DSIC-IR-Python.ipynb)) est un mécanisme
**prouvé sans faille de manipulation**. Ici, le théorème prouve le contraire — les règles du
jouet **ont** une faille — et la même technique formelle sert aux deux verdicts. La question
« ce mécanisme a-t-il un loophole ? » est littéralement décidable dans le modèle.

## 5. Ce que la recherche bornée ne sait pas — limites honnêtes

**Bornes de recherche.** Profondeur ≤ 4, montants ≤ plafond : « optimal » au §2 veut dire
*optimal dans cet espace fini*. Seul `roth_ceiling` transporte la conclusion à toute
profondeur — c'est précisément ce que la preuve ajoute au calcul.

**Hypothèses déclarées du modèle** (chacune est un écart assumé au droit réel — les trois
premières sont aussi les limites déclarées par l'article lui-même pour son modèle complet) :

| Hypothèse du jouet | Droit réel | Coût de l'écart |
|---|---|---|
| Pro-rata calculé **à chaque conversion** | agrégation en fin d'année (formulaire 8606, « plus permissive sur l'ordre ») | le jouet sur/sous-estime la taxe selon l'ordre des coups — exemple canonique d'hypothèse déclarée |
| Aucune règle de substance (§401(m) réelle) | l'IRS peut requalifier une conversion en « step transaction » | la faille du jouet est peut-être déjà fermée en pratique |
| Pas d'impôt d'État, pas de roll-in 401(k), pas de croissance, pas de MAGI | chaque règle absente change le plan optimal | aucune conclusion quantitative ne se transporte |
| Montants symboliques sur ℕ (1 unité = 1 000 $) | dollars réels, arrondis fiscaux propres | aucun chiffre de ce carnet ne se lit comme un montant |

**La faille n'est pas un bug.** Chaque coup du plan est légal au sens des règles données ; le
contournement est **dans les règles**, pas contre elles. Un détecteur d'erreurs chercherait
une transcription fausse du code fiscal — ici la transcription est fidèle et la faille reste :
c'est un **choix du législateur** (ou un accident qu'il n'a pas vu), pas une erreur mécanique.
D'où la question de conception du §6.

**Ce carnet n'est pas un conseil fiscal.**

## 6. Le jeu répété — le législateur ferme la faille

Les carnets [GameTheory-12](GameTheory-12-ReputationGames-Python.ipynb) et
[GameTheory-13e](GameTheory-13e-Revision-Croyances-Adversaire-Python.ipynb) ont étudié le jeu
répété : ce qui est optimal à une manche change quand le même adversaire revient et **apprend**.
Ici l'adversaire du contribuable est le **législateur** : si beaucoup de contribuables jouent le
backdoor, la règle bouge (l'article cite les débats autour de la *step transaction doctrine*
et des règles de substance type §401(m)). Le jeu des règles fiscales est donc un jeu **répété
à règles mobiles** : chaque théorème du §3 est valable pour une version figée des lois, et la
faille « prouvée existante » est une faille *de cette version*.

Formaliser le coup du législateur — ajouter une garde, changer le pro-rata — puis re-prouver,
est exactement le même geste que ce grain vient de faire pour la version initiale. C'est le
programme de recherche que l'article propose : le code fiscal comme objet formel versionné,
chaque version avec ses théorèmes de frontière.

## 7. Question directrice

> **Un loophole est-il une stratégie gagnante du jeu que le législateur a écrit sans le
> vouloir ?**

Le BFS répond « oui, en voici une » ; `roth_ceiling` répond « et voici jusqu'où elle peut
aller ». Qu'est-ce qui manque aux deux pour répondre « ... et le législateur peut la fermer » ?
(Indice : §6 — il manque le coup du législateur dans le jeu.)

## À retenir

- Une règle fiscale se lit comme un jeu formel : état, coups à gardes, sémantique exécutable
  (`step`/`run`).
- Une recherche bornée (BFS) retrouve un plan d'action **sans qu'on le lui décrive** — mais ses
  conclusions sont bornées par construction.
- Le compagnon Lean prouve l'universel : existence (`backdoor_exists`), plafond à toute
  profondeur (`roth_ceiling`), optimalité du témoin (`backdoor_optimal`). La méthode de
  l'article est **la paire recherche/preuve**.
- Les hypothèses d'un modèle juridique se **déclarent** (§5) : c'est la frontière entre une
  distillation honnête et un conseil déguisé.

## Vérifiez votre compréhension

**Question 1 — Qu'est-ce que `roth_ceiling` prouve que la BFS du §2 ne peut pas prouver ?**

La BFS rend `('nondeductible', 7)` puis `('convert', 7)`, un état final à `roth=7` et une taxe
totale de `0`, sur un espace de profondeur au plus 4.

<details><summary>Réponse</summary>

La BFS prouve un **témoin** : *ce* plan-là existe, il est légal, et il coûte `0`. Elle ne dit
rien des séquences plus profondes que 4, ni des montants hors plafond — « optimal » au §2 veut
dire *optimal dans cet espace fini*. `roth_ceiling` quantifie, lui, sur **toute** profondeur et
sur tout état atteignable : aucun plan, aussi long soit-il, ne porte le Roth au-delà du plafond
de contribution. C'est exactement ce que la preuve ajoute au calcul — et c'est ce qui autorise
l'article à qualifier son plan BFS d'optimal dans le modèle plutôt que de « meilleur trouvé ».

</details>

**Question 2 — Pourquoi la faille du jouet n'est-elle pas un bug, et que coûte-t-il de la
transposer au droit réel ?**

Le §5 énumère quatre écarts assumés entre le jouet et le droit réel : pro-rata à chaque
conversion, aucune règle de substance, pas d'impôt d'État ni de croissance, montants symboliques
sur ℕ.

<details><summary>Réponse</summary>

Parce que la transcription est **fidèle** : chaque coup du plan est légal au sens des règles
données, et le contournement est *dans* les règles, pas contre elles. Chercher un bug serait
chercher une erreur de transcription — il n'y en a pas ; il y a un choix du législateur (ou un
accident qu'il n'a pas vu). Les quatre écarts, eux, ne se compensent pas : le pro-rata par
conversion sur- ou sous-estime la taxe selon l'ordre des coups, l'absence de règle de substance
signifie que la faille est peut-être déjà fermée en pratique, et les montants symboliques
interdisent de lire un chiffre du carnet comme un montant. Aucune conclusion **quantitative** ne
se transporte — seule la forme du raisonnement se transporte.

</details>

**Question 3 — Le §6 change-t-il le statut des théorèmes du §3 ?**

Le §6 décrit le législateur comme l'adversaire du contribuable : si le backdoor est joué par
beaucoup, la règle bouge.

<details><summary>Réponse</summary>

Non, il en précise le **domaine de validité**. Chaque théorème du §3 porte sur une version
figée des lois ; la faille « prouvée existante » est une faille *de cette version*. Dès que le
législateur dispose d'un coup — ajouter une garde, changer le pro-rata — le jeu change et les
preuves doivent être rejouées sur la nouvelle version. C'est le programme que l'article propose :
le code fiscal comme objet formel **versionné**, chaque version avec ses théorèmes de frontière.
Le §3 ne devient donc pas faux ; il devient daté, ce qui est précisément le résultat honnête.

</details>

## Pour aller plus loin

- [Index de la série GameTheory](README.md) — la place de ce carnet parmi les autres structures
  de la théorie des jeux, et le fil des sous-séries.
- [Metajeux-05](Metajeux/Metajeux-05-Temoins-Impossibilite-Python.ipynb) — témoins et
  impossibilité : la même dualité, du côté de la preuve négative.
- [Lean-25](../SymbolicAI/Lean/Lean-25-Coherence-et-Temoin.ipynb) — cohérence et témoin, le
  vocabulaire formel employé ici.
- [GameTheory-16c](GameTheory-16c-Extraction-de-Revenu-DSIC-IR-Python.ipynb) — un mécanisme
  prouvé **sans** faille de manipulation : le contraste exact avec les règles du jouet.
- [GameTheory-13e](GameTheory-13e-Revision-Croyances-Adversaire-Python.ipynb) — l'adversaire
  apprend : ce que devient le §6 quand la règle est un joueur.
- Le module [`Toy.lean`](game_theory_lean/LegalLoophole/Toy.lean), sa racine
  `LegalLoophole.lean` et la carte [`LEAN_INVENTORY.md`](LEAN_INVENTORY.md) §1.